## 3. 텍스트의 수치화와 표현

**환경:** Google Colab / Jupyter Notebook
**준비물:** 단원 2에서 저장한 `cleaned_<엑셀파일명>.csv` (없어도 예제 리뷰로 진행 가능)


### 이 단원에서 하는 일
컴퓨터는 **글자를 계산하지 못하고 숫자만 계산**할 수 있습니다.
그래서 리뷰 문장을 **숫자 목록(벡터)** 으로 바꿔야 분석과 분류를 할 수 있습니다.
이 단원에서는 문장을 숫자로 바꾸는 방법 다섯 가지를 **쉬운 것부터 차례로** 배웁니다.

```
"배송 빠르다 포장 꼼꼼하다"  ──바꾸기──▶  [0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0]
```

### 전체 실습 진행 과정
| 절 | 주제 |  한 줄 설명 | 다음 절로 가는 이유 |
|---|---|---|---|
| 0 | 준비 |  데이터와 예제 리뷰 불러오기 | — |
| 3-1 | One-hot |  단어 하나를 "자리 하나만 1"인 숫자 목록으로 | 단어끼리 관계를 모름 |
| 3-2 | Bag-of-Words |  리뷰 하나를 "단어 개수" 목록으로 | 단어 순서를 모름 |
| 3-3 | 빈도·N-gram |  많이 나온 단어 세기, 붙어 나온 단어 묶기 | 흔한 단어가 너무 큼 |
| 3-4 | TF-IDF |  흔한 단어는 작게, 특별한 단어는 크게 | 리뷰끼리 비교는 아직 못 함 |
| 3-5 | 유사도 |  비슷한 리뷰 찾기 | 단어의 뜻은 모름 |
| 3-6 | 종합 비교 |  다섯 방법을 한눈에 비교 | 다음 단원 로 연결 |

## 꼭 알아둘 용어
| 용어 | 쉬운 뜻 | 예 |
|---|---|---|
| **벡터** | 숫자를 한 줄로 늘어놓은 목록 | `[0, 1, 1, 0]` |
| **차원** | 벡터 안 숫자의 개수 (= 칸 수) | `[0, 1, 1, 0]`은 4차원 |
| **단어 사전(vocabulary)** | 데이터에 나온 단어 목록과 번호 | `{"가격": 0, "배송": 1}` |
| **행렬** | 벡터를 여러 줄 쌓은 표 | 리뷰 5개 × 단어 11개 표 |
| **희소행렬** | 대부분이 0이라 0이 아닌 값만 저장한 행렬 | 메모리 절약용 |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`를 누르면 실행됩니다. **반드시 위에서부터 순서대로** 실행하세요.
> 중간부터 다시 시작할 때는 **0. 준비** 셀을 먼저 실행하세요.

---
## 0. 준비
- `cleaned_<엑셀파일명>.csv`를 Colab 왼쪽 **왼쪽 파일 창(📁)에 끌어다 놓으면** 그 데이터를 씁니다.
- 파일이 없으면 예제 리뷰로 진행하니 걱정하지 마세요.
- `mini`는 이 단원 **모든 절에서 함께 쓰는 리뷰 5개**입니다. 같은 문장이 방법마다 어떻게 바뀌는지 계속 지켜보세요.

| 번호 | mini 리뷰 | 눈여겨볼 점 |
|---|---|---|
| 0 | 배송 빠르다 포장 꼼꼼하다 | 칭찬 리뷰 |
| 1 | 배송 빠르다 포장 별로 | 1번과 2번은 **단어가 같고 순서만 다름** |
| 2 | 포장 빠르다 배송 별로 | (뜻은 서로 다름) |
| 3 | 가격 저렴하다 품질 좋다 | 3번과 4번은 **뜻이 반대** |
| 4 | 가격 비싸다 품질 안 좋다 | (`좋다` vs `안 좋다`) |

-   Colab 왼쪽 **📁 파일 창**에 `cleaned_<엑셀파일명>.csv`를 끌어다 놓으세요.




In [7]:
# 파일 찾기 도구 불러오기
import glob
# 숫자 계산 도구 불러오기
import numpy as np
# 표 다루기 도구 불러오기
import pandas as pd

# 모든 절에서 함께 쓸 예제 리뷰 5개 (단원 2처럼 토큰화가 끝난 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

# 'cleaned_'로 시작하는 CSV 파일 찾기
files = glob.glob("cleaned_*.csv")

# docs 세팅
# 파일이 있으면 그 파일의 토큰문자열을 사용
if files:
    data = pd.read_csv(files[0])                          # CSV 읽기
    docs = data["토큰문자열"].dropna().astype(str).tolist()  # 빈 값 빼고 목록으로 만들기
    docs = [d for d in docs if d.strip()]                 # 공백만 있는 리뷰 빼기
    print("데이터 파일:", files[0])
# 파일이 없으면 예제 리뷰를 20번 반복해서 사용
else:
    docs = mini * 20
    print("[안내] 단원 2 CSV가 없어 예제 리뷰로 진행합니다.")

# 리뷰 개수 출력
print("리뷰 수:", len(docs))


데이터 파일: cleaned_customer_inquiries.csv
리뷰 수: 3001


---
## 3-1. One-hot Encoding (원-핫 인코딩)

### 🎯 이 절에서 배울 것
- 단어 하나를 숫자 목록으로 바꾸는 **가장 단순한 방법**
- 이 방법의 문제점

### 💡 비유: 출석부
반 학생이 11명이면 출석부에 11칸이 있습니다. "배송"이라는 학생을 표시할 때는 **배송 칸에만 ○**, 나머지는 모두 빈칸입니다.
One-hot은 이처럼 **단어 자리 하나만 1이고 나머지는 모두 0**인 숫자 목록입니다.

| 단계 | 내용 |
|---|---|
| 1 | 단어 사전(출석부) 만들기 |
| 2 | 단어 하나를 one-hot으로 |
| 3 | 문장을 one-hot 표로 |
| 4 | 문제점: 단어끼리 관계를 모름 |


### 1단계. 단어 사전 만들기
리뷰에 나온 단어를 **겹치지 않게** 모아 번호를 붙입니다. 단어가 몇 개인지가 곧 벡터의 칸 수(차원)입니다.

In [8]:
# 모든 절에서 함께 쓸 예제 리뷰 5개 (단원 2처럼 토큰화가 끝난 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]
# 리뷰 5개를 한 문장으로 이어 붙이기
all_text = " ".join(mini)

# 띄어쓰기로 단어를 나누고, set으로 중복 제거, sorted로 가나다순 정렬
vocab = sorted(set(all_text.split()))

print("vocab:",vocab)

# 단어 사전 크기 출력
print("단어 사전 크기:", len(vocab))

# 단어마다 번호 붙여서 출력
for number, word in enumerate(vocab):
    print(number, word)

vocab: ['가격', '꼼꼼하다', '배송', '별로', '비싸다', '빠르다', '안', '저렴하다', '좋다', '포장', '품질']
단어 사전 크기: 11
0 가격
1 꼼꼼하다
2 배송
3 별로
4 비싸다
5 빠르다
6 안
7 저렴하다
8 좋다
9 포장
10 품질


### 2단계. 단어 하나를 one-hot으로
"배송"의 one-hot 벡터를 만들어 봅니다. 사전에서 **배송 자리만 1**입니다.

In [9]:
print("vocab:",vocab)

# 사전의 단어를 하나씩 보면서, '배송'이면 1 아니면 0
one_hot_배송 = [1 if w == "배송" else 0 for w in vocab]

# 결과 출력
print(one_hot_배송)

vocab: ['가격', '꼼꼼하다', '배송', '별로', '비싸다', '빠르다', '안', '저렴하다', '좋다', '포장', '품질']
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]


### 3단계. 문장을 one-hot 표로
문장의 단어마다 one-hot을 만들어 쌓으면 **단어 수 × 사전 크기** 표가 됩니다.

In [14]:
# 첫 번째 리뷰를 단어로 나누기
print("첫 번째 리뷰:",mini[0])
print("vocab:",vocab)
words = mini[0].split()

# 단어마다 one-hot 벡터를 만들어 목록에 담기
rows = []
for word in words:
    rows.append([1 if w == word else 0 for w in vocab])

print("rows:", rows)

# 보기 좋게 표로 출력 (행: 리뷰의 단어, 열: 사전 단어)
pd.DataFrame(rows, index=words, columns=vocab)

첫 번째 리뷰: 배송 빠르다 포장 꼼꼼하다
vocab: ['가격', '꼼꼼하다', '배송', '별로', '비싸다', '빠르다', '안', '저렴하다', '좋다', '포장', '품질']
rows: [[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


,가격,꼼꼼하다,배송,별로,비싸다,빠르다,안,저렴하다,좋다,포장,품질
배송,0,0,1,0,0,0,0,0,0,0,0
빠르다,0,0,0,0,0,1,0,0,0,0,0
포장,0,0,0,0,0,0,0,0,0,1,0
꼼꼼하다,0,1,0,0,0,0,0,0,0,0,0


✅ 줄마다 **1이 딱 하나**인 것을 확인하세요.

### 4단계. 문제점: 단어끼리 관계를 모름
두 벡터의 **같은 칸끼리 곱해서 더한 값(내적)** 으로 얼마나 닮았는지 잴 수 있습니다.
One-hot은 1의 위치가 단어마다 달라서, **어떤 두 단어를 비교해도 0**이 나옵니다.



In [20]:
print("vocab:",vocab)

# 단어를 넣으면 one-hot 벡터를 돌려주는 함수
#파이썬 리스트끼리는 @ 연산을 할 수 없습니다. NumPy 배열로 바꿔야 벡터 연산(내적)을 할 수 있음
def one_hot(word):
    return np.array([1 if w == word else 0 for w in vocab])

print('one_hot(저렴하다) :',one_hot("저렴하다"))

# 반대말 쌍의 내적
#@는 행렬 곱셈 연산자입니다. 두 1차원 배열에 쓰면 내적(dot product), 즉 같은 위치의 값끼리 곱해서 모두 더한 값 하나가 나옵니다.
print("저렴하다 · 비싸다 =", one_hot("저렴하다") @ one_hot("비싸다"))

# 관계없는 쌍의 내적
print("좋다 · 꼼꼼하다 =", one_hot("좋다") @ one_hot("꼼꼼하다"))

vocab: ['가격', '꼼꼼하다', '배송', '별로', '비싸다', '빠르다', '안', '저렴하다', '좋다', '포장', '품질']
one_hot(저렴하다) : [0 0 0 0 0 0 0 1 0 0 0]
저렴하다 · 비싸다 = 0
좋다 · 꼼꼼하다 = 0


✅ 반대말이든 비슷한 말이든 모두 0입니다. One-hot은 **단어 사이의 관계를 전혀 표현하지 못합니다.**

### [참조] 비슷한 이름의 다른 방법 구분하기
| 이름 | 무엇을 표현 | 1의 개수 | 예 |
|---|---|---|---|
| **One-hot** | 단어 1개 | 딱 1개 | `배송` → `[0,0,1,0,…]` |
| **Binary BoW** | 리뷰 1개 | 리뷰에 나온 단어 수만큼 | `배송 빠르다 포장` → `[0,1,1,1,…]` |

⚠️ 실무에서는 리뷰 단위 0/1 벡터도 "원-핫"이라고 부르는 경우가 많아 헷갈리기 쉽습니다. 엄밀히는 **Binary BoW**(또는 multi-hot)입니다.

In [22]:
# 단어 개수 세기 도구 불러오기
from sklearn.feature_extraction.text import CountVectorizer

# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# binary=True: 개수 대신 '나왔으면 1, 안 나왔으면 0'
cv = CountVectorizer(binary=True, **opts)

# 예제 리뷰 5개를 0/1 벡터로 바꾸기
X = cv.fit_transform(mini)

# 표로 출력 (행: 리뷰, 열: 단어)
pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out())

,가격,꼼꼼하다,배송,별로,비싸다,빠르다,안,저렴하다,좋다,포장,품질
0,0,1,1,0,0,1,0,0,0,1,0
1,0,0,1,1,0,1,0,0,0,1,0
2,0,0,1,1,0,1,0,0,0,1,0
3,1,0,0,0,0,0,0,1,1,0,1
4,1,0,0,0,1,0,1,0,1,0,1



### ✅ 3-1 한 줄 정리
**One-hot = 단어 자리 하나만 1.** 간단하지만 칸이 너무 많고, 단어끼리 관계를 모릅니다.

### ➡️ 다음 절로
One-hot은 **단어 하나**를 표현했습니다. 하지만 우리가 분석할 것은 **리뷰 한 개 전체**입니다.
3-2절에서는 리뷰 하나를 **숫자 목록 하나**로 바꾸는 방법을 배웁니다.

---
## 3-2. Bag-of-Words (BoW, 단어 가방)

### 🎯 이 절에서 배울 것
- 리뷰 하나를 **단어 개수 목록**으로 바꾸기
- 초보자가 가장 많이 걸리는 **한 글자 단어 함정**
- BoW의 문제점: **순서를 모른다**

### 💡 비유: 장바구니 영수증
장을 보면 영수증에 "사과 2개, 우유 1개"처럼 **무엇을 몇 개** 샀는지만 적히고, **어떤 순서로 담았는지**는 적히지 않습니다.
BoW는 리뷰를 장바구니처럼 보고 **단어별 개수만** 적습니다.

| 단계 | 내용 |
|---|---|
| 1 | BoW 표 만들기 |
| 2 | 단어 사전 보기 |
| 3 | 한 글자 단어 함정 |
| 4 | 문제점: 순서를 모름 |
| 5 | 실제 데이터와 희소행렬 |
| 6 | 새 문장 바꾸기 |

### 1단계. BoW 표 만들기
scikit-learn의 `CountVectorizer`가 **단어 나누기 + 개수 세기**를 한 번에 해 줍니다.
- `fit_transform()`: 단어 사전을 만들고(`fit`), 리뷰를 숫자로 바꾸기(`transform`)를 한 번에

In [28]:
# 단어 개수 세기 도구 불러오기
from sklearn.feature_extraction.text import CountVectorizer

# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 개수 세기 도구 만들기 (공통 설정 opts 사용)
# **opts : 딕셔너리를 펼쳐 키=값 형태의 인자로 넘김
cv = CountVectorizer(**opts)

# 모든 절에서 함께 쓸 예제 리뷰 5개 (단원 2처럼 토큰화가 끝난 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

# 사전 만들기 + 리뷰를 숫자로 바꾸기
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬
# X         : 행 = 리뷰, 열 = 단어, 값 = 횟수인 희소 행렬
X = cv.fit_transform(mini)


# 표로 출력 (행: 리뷰, 열: 단어, 값: 개수)
pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=mini)

,가격,꼼꼼하다,배송,별로,비싸다,빠르다,안,저렴하다,좋다,포장,품질
배송 빠르다 포장 꼼꼼하다,0,1,1,0,0,1,0,0,0,1,0
배송 빠르다 포장 별로,0,0,1,1,0,1,0,0,0,1,0
포장 빠르다 배송 별로,0,0,1,1,0,1,0,0,0,1,0
가격 저렴하다 품질 좋다,1,0,0,0,0,0,0,1,1,0,1
가격 비싸다 품질 안 좋다,1,0,0,0,1,0,1,0,1,0,1


### 2단계. 단어 사전 보기
`vocabulary_`에는 **단어 → 몇 번째 칸** 정보가 들어 있습니다.

In [29]:
## fit  : 5개 리뷰에 나온 모든 단어를 모아 단어 사전(vocabulary_)을 만듦
# 단어별 칸 번호 출력
cv.vocabulary_

{'배송': 2,
 '빠르다': 5,
 '포장': 9,
 '꼼꼼하다': 1,
 '별로': 3,
 '가격': 0,
 '저렴하다': 7,
 '품질': 10,
 '좋다': 8,
 '비싸다': 4,
 '안': 6}

### 3단계. ⚠️ 한 글자 단어 함정
`CountVectorizer()`를 **설정 없이** 쓰면 공식 문서 기준으로 **2글자 이상인 단어만** 남깁니다.
그래서 `안`, `향`, `못` 같은 **한 글자 단어가 조용히 사라집니다.** 부정어 `안`이 사라지면 "안 좋다"가 "좋다"가 됩니다!

In [31]:
# 설정 없이 만든 사전
print("설정 없음:", CountVectorizer().fit(mini).get_feature_names_out())

# 공통 설정(opts)으로 만든 사전
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)
print("opts 사용:", CountVectorizer(**opts).fit(mini).get_feature_names_out())

설정 없음: ['가격' '꼼꼼하다' '배송' '별로' '비싸다' '빠르다' '저렴하다' '좋다' '포장' '품질']
opts 사용: ['가격' '꼼꼼하다' '배송' '별로' '비싸다' '빠르다' '안' '저렴하다' '좋다' '포장' '품질']


✅ 설정 없이 만든 사전에는 `안`이 없습니다. **토큰화가 끝난 한국어 데이터에는 꼭 `opts`를 쓰세요.**

### 4단계. 문제점: 순서를 모름
- `mini[1]` = 배송 빠르다 포장 별로 → 배송은 좋고 포장이 별로
- `mini[2]` = 포장 빠르다 배송 별로 → 포장은 좋고 배송이 별로

뜻이 다른 두 리뷰가 BoW에서는 어떻게 될까요?

In [32]:
# 두 리뷰의 BoW 벡터 출력
print(mini[1], "→", X[1].toarray()[0])
print(mini[2], "→", X[2].toarray()[0])

# 두 벡터가 완전히 같은지 확인 (다른 칸 수가 0이면 같음)
print("같은 벡터인가?", (X[1] != X[2]).nnz == 0)

배송 빠르다 포장 별로 → [0 0 1 1 0 1 0 0 0 1 0]
포장 빠르다 배송 별로 → [0 0 1 1 0 1 0 0 0 1 0]
같은 벡터인가? True


✅ 뜻이 다른데 **완전히 같은 벡터**입니다. 장바구니 영수증처럼 **무엇이 몇 개**인지만 알고 **순서는 모릅니다.**

### 5단계. 실제 데이터와 희소행렬
실제 리뷰 수천 개로 BoW를 만들면 표가 엄청 커지고 **거의 다 0**입니다.
그래서 scikit-learn은 **0이 아닌 값만 저장**하는 희소행렬로 돌려줍니다.

> ⚠️ 실제 데이터에는 `.toarray()`를 쓰지 마세요! 0까지 다 저장하면 메모리가 크게 늘어 Colab이 멈출 수 있습니다.

In [34]:
# 실제 데이터로 BoW 만들기
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

#docs =엑셀파일 또는 mini X 20
cv_real = CountVectorizer(**opts)
X_real = cv_real.fit_transform(docs)
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬

# 표 크기 (리뷰 수, 단어 수)
rows, cols = X_real.shape
print("표 크기(리뷰 수, 단어 수):", X_real.shape)

# 0이 아닌 칸 수
print("0이 아닌 칸 수:", X_real.nnz)

# 전체 칸 중 0의 비율
print("0의 비율(%):", round((1 - X_real.nnz / (rows * cols)) * 100, 2))

# 0까지 모두 저장하면 필요한 메모리 (칸 하나에 8바이트로 계산)
print("모두 저장하면 약", round(rows * cols * 8 / 1024**2), "MB")

표 크기(리뷰 수, 단어 수): (3001, 1775)
0이 아닌 칸 수: 42342
0의 비율(%): 99.21
모두 저장하면 약 41 MB


### 6단계. 새 문장 바꾸기
이미 만든 사전으로 새 문장을 바꿀 때는 `transform()`만 씁니다. **사전에 없는 단어는 버려집니다.**

In [36]:
# 사전에 없는 '배터리'가 들어간 새 문장 바꾸기
new = cv.transform(["배송 빠르다 배터리 좋다"])

# 표로 출력 (배터리 칸은 없음)
pd.DataFrame(new.toarray(), columns=cv.get_feature_names_out())

,가격,꼼꼼하다,배송,별로,비싸다,빠르다,안,저렴하다,좋다,포장,품질
0,0,0,1,0,0,1,0,0,1,0,0


### ✅ 3-2 한 줄 정리
**BoW = 리뷰마다 단어 개수 세기.** 쉽고 빠르지만 **순서를 모르고**, 흔한 단어가 큰 값을 가집니다.

### ➡️ 다음 절로
3-3절에서는 **어떤 단어가 많이 나오는지** 세어 보고, **붙어 나오는 단어 묶음(N-gram)** 으로 순서 정보를 일부 되살립니다.

---
## 3-3. 단어 빈도와 N-gram

### 🎯 이 절에서 배울 것
- 가장 많이 나온 단어 찾기
- "몇 번 나왔나"와 "몇 개 리뷰에 나왔나"의 차이
- N-gram: 붙어 나온 단어를 **묶음**으로 세기

### 💡 비유: 2글자 낱말 카드
"안"과 "좋다"를 따로 세면 뜻이 사라지지만, **"안 좋다"를 한 장의 카드**로 만들면 뜻이 남습니다. 이렇게 연속된 N개 단어를 묶은 것이 **N-gram**입니다.

| 이름 | 묶는 단어 수 | 예 (`품질 안 좋다`) |
|---|---|---|
| unigram (1-gram) | 1개 | `품질`, `안`, `좋다` |
| bigram (2-gram) | 2개 | `품질 안`, `안 좋다` |
| trigram (3-gram) | 3개 | `품질 안 좋다` |

| 단계 | 내용 |
|---|---|
| 1 | 많이 나온 단어 |
| 2 | 총빈도 vs 문서 빈도 |
| 3 | N-gram 만들기 |
| 4 | 실제 데이터에서 많이 붙어 나온 단어 |

### 1단계. 많이 나온 단어
BoW 표를 **세로로 모두 더하면** 단어별로 전체에서 몇 번 나왔는지 알 수 있습니다.

In [52]:
# 단어 개수 세기 도구 불러오기
from sklearn.feature_extraction.text import CountVectorizer

# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

#docs =엑셀파일 또는 mini X 20

# 실제 데이터로 BoW 만들기
cv = CountVectorizer(**opts)
X = cv.fit_transform(docs)
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬
# X         : 행 = 리뷰, 열 = 단어, 값 = 횟수인 희소 행렬

# 세로 방향으로 더해서 단어별 전체 횟수 구하기
s = X.sum(axis=0)        # 1단계: 열(단어)마다 모든 행(리뷰)의 값을 더함
a = np.asarray(s)        # 2단계: 결과를 일반 NumPy 배열(ndarray)로 바꿈
total = a.ravel()        # 3단계: 몇 차원이든 1차원으로 펼침

# 단어 이름을 붙여 표로 만들기
freq = pd.Series(total, index=cv.get_feature_names_out())

# 많이 나온 순서로 15개 출력
freq.sort_values(ascending=False).head(15)

,0
좋다,3531
사용,1994
제품,1739
향,1429
같다,1346
쓰다,1335
않다,1184
구매,1092
바르다,1038
하다,970


### 2단계. 총빈도 vs 문서 빈도
| 이름 | 뜻 | 예 |
|---|---|---|
| **총빈도** | 전체에서 **몇 번** 나왔나 | 한 리뷰에 "좋다"가 3번 나오면 3 |
| **문서 빈도** | **몇 개 리뷰**에 나왔나 | 같은 경우 1 |

문서 빈도는 다음 절 **TF-IDF의 재료**입니다.

In [54]:
# 나왔으면 1, 안 나왔으면 0으로 세기
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)
#docs =엑셀파일 또는 mini X 20
X_bin = CountVectorizer(binary=True, **opts).fit_transform(docs)
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬


# 총빈도와 문서 빈도를 한 표에 담기
table = pd.DataFrame({
    "총빈도": np.asarray(X.sum(axis=0)).ravel(),        # 열마다 합계를 구해 1차원 배열로 바꿈 → 전체 등장 횟수
    "문서빈도": np.asarray(X_bin.sum(axis=0)).ravel(),  # 0/1 행렬의 열 합계 → 등장한 리뷰 수
}, index=cv.get_feature_names_out())

# 문서 비율(%) = 전체 리뷰 중 몇 %에 나왔나
table["문서비율(%)"] = (table["문서빈도"] / len(docs) * 100).round(1)

# 총빈도 순으로 15개 출력
table.sort_values("총빈도", ascending=False).head(15)

,총빈도,문서빈도,문서비율(%)
좋다,3531,2171,72.3
사용,1994,1431,47.7
제품,1739,1261,42.0
향,1429,1167,38.9
같다,1346,1074,35.8
쓰다,1335,1014,33.8
않다,1184,942,31.4
구매,1092,923,30.8
바르다,1038,757,25.2
하다,970,745,24.8


✅ 가장 많이 나온 단어는 **절반 가까운 리뷰**에 나옵니다. 이렇게 흔한 단어는 리뷰를 **구별하는 데 도움이 안 됩니다.**

### 3단계. N-gram 만들기
`ngram_range=(1, 2)`는 "한 단어와 두 단어 묶음을 함께 세라"는 뜻입니다.

In [58]:
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 1단어 + 2단어 묶음으로 세기
cv2 = CountVectorizer(ngram_range=(1, 2), **opts)

# 모든 절에서 함께 쓸 예제 리뷰 5개 (단원 2처럼 토큰화가 끝난 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

# 예제 리뷰 5개 바꾸기
X2 = cv2.fit_transform(mini)
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬


# 표로 출력
pd.DataFrame(X2.toarray(), columns=cv2.get_feature_names_out(), index=mini)

,가격,가격 비싸다,가격 저렴하다,꼼꼼하다,배송,배송 별로,배송 빠르다,별로,비싸다,비싸다 품질,...,저렴하다,저렴하다 품질,좋다,포장,포장 꼼꼼하다,포장 별로,포장 빠르다,품질,품질 안,품질 좋다
배송 빠르다 포장 꼼꼼하다,0,0,0,1,1,0,1,0,0,0,...,0,0,0,1,1,0,0,0,0,0
배송 빠르다 포장 별로,0,0,0,0,1,0,1,1,0,0,...,0,0,0,1,0,1,0,0,0,0
포장 빠르다 배송 별로,0,0,0,0,1,1,0,1,0,0,...,0,0,0,1,0,0,1,0,0,0
가격 저렴하다 품질 좋다,1,0,1,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,1,0,1
가격 비싸다 품질 안 좋다,1,1,0,0,0,0,0,0,1,1,...,0,0,1,0,0,0,0,1,1,0


✅ 확인할 점
- `mini[4]`에 **`안 좋다`** 칸이 생겼습니다 → 부정의 뜻이 살아남!
- `mini[1]`은 `배송 빠르다`, `mini[2]`는 `포장 빠르다` → 3-2절에서 같았던 두 리뷰가 **이제 다른 벡터**입니다.

### 4단계. 실제 데이터에서 많이 붙어 나온 단어

In [59]:
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)
# 2단어 묶음만 세기
cv_bi = CountVectorizer(ngram_range=(2, 2), **opts)

X_bi = cv_bi.fit_transform(docs) #docs =엑셀파일 또는 mini X 20
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬


# 묶음별 전체 횟수 구하기
bigram = pd.Series(
    np.asarray(X_bi.sum(axis=0)).ravel(),   # 열(2-gram)마다 합계를 구해 1차원 배열로 바꿈 → 2-gram별 전체 등장 횟수
    index=cv_bi.get_feature_names_out()     # 각 값에 2-gram 이름(예: '배송 빠르다')을 붙임
)

# 많이 나온 순서로 15개 출력
bigram.sort_values(ascending=False).head(15)

,0
좋다 같다,378
향 좋다,374
올 인원,317
사용 좋다,261
배송 빠르다,248
좋다 하다,245
사용 제품,222
가격 저렴하다,212
좋다 않다,200
제품 사용,194


🤔 **생각해 볼 점:** 단원 2에서 조사·어미를 지웠기 때문에, 원래 문장에서는 떨어져 있던 단어가 붙어 묶음이 되기도 합니다.
예: "배송**이** 빨라**서**" → `배송 빠르다`. 의미 있는 묶음인지 원래 리뷰와 비교해 보세요.


### ✅ 3-3 한 줄 정리
**빈도 = 많이 나온 단어 세기, N-gram = 붙어 나온 단어 묶기.** 순서 정보가 일부 살아나지만 칸이 크게 늘어납니다.

### ➡️ 다음 절로
많이 나온 단어 상위에는 **어느 리뷰에나 나오는 흔한 단어**가 올라옵니다. 3-4절 **TF-IDF**는 흔한 단어의 점수를 **낮추고**, 그 리뷰만의 특별한 단어 점수를 **높입니다.**

---
## 3-4. TF-IDF

### 🎯 이 절에서 배울 것
- TF-IDF가 단어 점수를 매기는 원리
- 직접 계산한 값과 scikit-learn 결과 비교
- 리뷰마다 **핵심 키워드** 뽑기

### 💡 비유: 반장 선거 공약
모든 후보가 "열심히 하겠습니다"라고 말하면 그 말은 후보를 구별해 주지 못합니다. 한 후보만 "급식 메뉴 투표제"를 말했다면 그게 **그 후보의 특징**입니다.
TF-IDF는 **이 리뷰에 많이 나오고(TF)** + **다른 리뷰에는 잘 안 나오는(IDF)** 단어에 높은 점수를 줍니다.

| 이름 | 뜻 | 커지는 경우 |
|---|---|---|
| **TF** (단어 빈도) | 이 리뷰에 몇 번 나왔나 | 이 리뷰에 많이 나올수록 |
| **IDF** (역문서 빈도) | 얼마나 드문 단어인가 | **적은 리뷰에만** 나올수록 |
| **TF-IDF** | TF × IDF | 이 리뷰에만 특별히 많이 나올수록 |

| 단계 | 내용 |
|---|---|
| 1 | IDF 공식 |
| 2 | IDF 직접 계산 |
| 3 | TF-IDF 표 보기 |
| 4 | 리뷰별 핵심어 |
| 5 | 실제 데이터의 흔한 단어·드문 단어 |
| 6 | 리뷰별 키워드 저장 |

### 1단계. IDF 공식
scikit-learn이 쓰는 공식(공식 문서 기준, 기본 설정)은 다음과 같습니다.

```
IDF = ln( (1 + 전체 리뷰 수) / (1 + 그 단어가 나온 리뷰 수) ) + 1
```

- 많은 리뷰에 나온 단어 → 분모가 커짐 → **IDF 작아짐**
- 적은 리뷰에 나온 단어 → 분모가 작아짐 → **IDF 커짐**
- 마지막에 리뷰마다 숫자 목록의 길이를 1로 맞추는 **정규화**를 합니다.

> ⚠️ **교재마다 공식이 조금씩 다릅니다.** `+1`이 없는 공식도 많아서 교재 공식으로 계산하면 scikit-learn과 숫자가 다릅니다. 이 실습은 scikit-learn 공식을 따릅니다.

### 2단계. IDF 직접 계산
예제 리뷰 5개로 직접 계산해서 scikit-learn 결과와 같은지 확인합니다.

In [82]:
# TF-IDF 도구 불러오기
from sklearn.feature_extraction.text import TfidfVectorizer
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)
# TF-IDF 도구 만들고 예제 리뷰로 학습
tv = TfidfVectorizer(**opts)

# 모든 절에서 함께 쓸 예제 리뷰 5개 (단원 2처럼 토큰화가 끝난 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]


X = tv.fit_transform(mini)
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬

# 사전 단어 목록
words = tv.get_feature_names_out()

# 전체 리뷰 수
n = len(mini)

# 단어마다 '몇 개 리뷰에 나왔나' 세기
df = []
for w in words:
    count = 0
    for review in mini:
        if w in review.split():   # 리뷰에 그 단어가 있으면
            count += 1            # 1 더하기
    df.append(count)
df = np.array(df)

# 공식대로 직접 계산한 IDF
my_idf = np.log((1 + n) / (1 + df)) + 1

# 직접 계산한 값과 scikit-learn 값을 나란히 출력
pd.DataFrame({"나온 리뷰 수": df, "직접 계산": my_idf, "scikit-learn": tv.idf_}, index=words).round(3)

,나온 리뷰 수,직접 계산,scikit-learn
가격,2,1.693,1.693
꼼꼼하다,1,2.099,2.099
배송,3,1.405,1.405
별로,2,1.693,1.693
비싸다,1,2.099,2.099
빠르다,3,1.405,1.405
안,1,2.099,2.099
저렴하다,1,2.099,2.099
좋다,2,1.693,1.693
포장,3,1.405,1.405


✅ 두 열이 **똑같습니다.** 리뷰 3개에 나온 `배송`·`포장`은 IDF가 낮고, 1개에만 나온 `꼼꼼하다`·`안`은 높습니다.

### 3단계. TF-IDF 표 보기

In [63]:
# 리뷰 × 단어 TF-IDF 점수 표 (소수 둘째 자리까지)
pd.DataFrame(X.toarray(), columns=words, index=mini).round(2)

,가격,꼼꼼하다,배송,별로,비싸다,빠르다,안,저렴하다,좋다,포장,품질
배송 빠르다 포장 꼼꼼하다,0.00,0.65,0.44,0.00,0.0,0.44,0.0,0.00,0.00,0.44,0.00
배송 빠르다 포장 별로,0.00,0.00,0.47,0.57,0.0,0.47,0.0,0.00,0.00,0.47,0.00
포장 빠르다 배송 별로,0.00,0.00,0.47,0.57,0.0,0.47,0.0,0.00,0.00,0.47,0.00
가격 저렴하다 품질 좋다,0.47,0.00,0.00,0.00,0.0,0.00,0.0,0.58,0.47,0.00,0.47
가격 비싸다 품질 안 좋다,0.41,0.00,0.00,0.00,0.5,0.00,0.5,0.00,0.41,0.00,0.41


### 4단계. 리뷰별 핵심어
리뷰마다 **점수가 가장 큰 단어**가 그 리뷰의 핵심어입니다.

In [83]:
# 리뷰마다 점수 상위 2개 단어 출력
for i in range(len(mini)):
    scores = X[i].toarray()[0]           # i번째 리뷰의 점수 목록
    top2 = scores.argsort()[::-1][:2]    # 점수 큰 순서로 칸 번호 2개
    # argsort()   : 값이 작은 순서대로 정렬했을 때의 "칸 번호"를 돌려줌
    # [::-1]      : 순서를 뒤집음 → 큰 값의 칸 번호부터
    # [:2]        : 앞에서 2개만 잘라냄
    print(mini[i], "→", list(words[top2]))

배송 빠르다 포장 꼼꼼하다 → ['꼼꼼하다', '빠르다']
배송 빠르다 포장 별로 → ['별로', '빠르다']
포장 빠르다 배송 별로 → ['별로', '빠르다']
가격 저렴하다 품질 좋다 → ['저렴하다', '좋다']
가격 비싸다 품질 안 좋다 → ['안', '비싸다']


### 5단계. 실제 데이터의 흔한 단어·드문 단어

In [84]:
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 실제 데이터로 TF-IDF 학습 (5개 미만 리뷰에 나온 단어는 제외)
#docs =엑셀파일 또는 mini X 20
tv_real = TfidfVectorizer(min_df=5, **opts)
X_real = tv_real.fit_transform(docs)
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬

# 단어별 IDF를 작은 순서로 정렬
# tv_real.idf_                     : fit할 때 계산된 단어별 IDF 값 배열 (단어 사전 순서와 같음)
# tv_real.get_feature_names_out()  : 같은 순서의 단어 이름 배열
# pd.Series(값, index=이름)         : IDF 값에 단어 이름표를 붙인 Series를 만듦
# .sort_values()                   : 값이 작은 순서(오름차순)로 정렬함
idf = pd.Series(tv_real.idf_, index=tv_real.get_feature_names_out()).sort_values()

# IDF가 낮은(흔한) 단어 10개
print("흔한 단어:", idf.head(10))

# IDF가 높은(드문) 단어 10개
print("드문 단어:", idf.tail(10))

흔한 단어: 좋다     1.323630
사용     1.740207
제품     1.866581
향      1.943986
같다     2.026958
쓰다     2.084390
않다     2.157968
구매     2.178322
가격     2.369776
바르다    2.376351
dtype: float64
드문 단어: 단계     7.215275
다행     7.215275
나라     7.215275
꺼리다    7.215275
기본     7.215275
국산     7.215275
교체     7.215275
계속     7.215275
결과     7.215275
거칠다    7.215275
dtype: float64


🤔 **생각해 볼 점:** IDF가 흔한 단어의 점수를 자동으로 낮춰 주니, "단원 2에서 불용어를 꼭 지워야 했을까?"는 **의견이 갈리는 문제**입니다. scikit-learn 공식 문서도 불용어 목록은 신중하게 쓰라고 안내합니다.

### 6단계. 리뷰별 키워드 저장
모든 리뷰에서 점수 상위 3개 단어를 뽑아 CSV 파일로 저장합니다.

In [85]:
# 실제 데이터 사전 단어 목록
# tv_real.get_feature_names_out()  : 같은 순서의 단어 이름 배열
real_words = tv_real.get_feature_names_out()

# 리뷰마다 키워드를 담을 목록
keywords = []
for i in range(X_real.shape[0]):
    scores = X_real[i].toarray()[0]               # i번째 리뷰 점수
    top = scores.argsort()[::-1][:3]              # 점수 큰 칸 번호 3개
    top = [t for t in top if scores[t] > 0]       # 점수 0인 칸은 빼기
    keywords.append(" ".join(real_words[top]))    # 단어를 띄어쓰기로 연결

# 리뷰와 키워드를 표로 만들기 #docs =엑셀파일 또는 mini X 20
result = pd.DataFrame({"리뷰토큰": docs, "TF-IDF키워드": keywords})

# CSV 파일로 저장 (한글 깨짐 방지 인코딩)
result.to_csv("tfidf_keywords.csv", index=False, encoding="utf-8-sig")

# 앞부분 10개 출력
result.head(10)

,리뷰토큰,TF-IDF키워드
0,남편 사다 광고 말 향 않다 올드 남자 스킨 로션 향 향,올드 향 말
1,전 쓰다 좋다 같다 향 독하다 피부 자극 강하다,독하다 전 강하다
2,남편 선물 사다 향기 좋다 지인 추천 택배 상자 안 상자 찌그러지다 오다 깨지다 않...,상자 택배 찌그러지다
3,스킨 로션 구성 세트 가성비 좋다 스킨 부족 구성 좋다 향 좋다 부드러,구성 스킨 세트
4,좋아하다 화장품 효과 좋다 화장품 형 농도 짙다 감기다 끈적이다 없다 흡수 좋다,화장품 짙다 형
5,향 남자 선호 스타일 명품 화장품 세트 득템,스타일 세트 선호
6,받다 향기 주문 안 하다 싸구려 향기 사우나 요즘 안 쓰다 향 싸구려 쎈 향기 뜯다...,싸구려 향기 어쩌다
7,가격 저렴하다 용량 타사 비하다 많다 차음 쓰다,타사 비하다 많다
8,빠르다 배송 여름 다가오다 스킨 필요 향기 좋다 제품 구성 만족 제품 용량 작다,필요 구성 향기
9,남자 화장품 향 강하다 편 많다 기능 향 좋다,기능 남자 향



</details>

### ✅ 3-4 한 줄 정리
**TF-IDF = 이 리뷰에 많고 + 다른 리뷰엔 드문 단어에 높은 점수.** 리뷰별 핵심 키워드를 뽑을 수 있습니다.

### ➡️ 다음 절로
리뷰를 점수 목록(벡터)으로 만들었으니, 이제 **두 리뷰가 얼마나 비슷한지** 숫자로 잴 수 있습니다. 3-5절에서 비슷한 리뷰를 찾아봅니다.

---
## 3-5. 벡터 기반 텍스트 표현과 유사도

### 🎯 이 절에서 배울 것
- 두 리뷰가 얼마나 비슷한지 숫자로 재기 (**코사인 유사도**)
- 비슷한 리뷰 찾기, 검색어로 리뷰 찾기
- 지금까지 배운 방법의 **근본적인 한계**

### 💡 비유: 화살표 방향
리뷰 벡터를 공간 속 **화살표**로 생각하세요. 두 화살표가 **같은 방향**을 가리키면 내용이 비슷한 것입니다.

| 코사인 유사도 | 뜻 |
|---|---|
| 1 | 방향이 같음 → 매우 비슷 |
| 0 | 직각 → 겹치는 단어가 없음 |

TF-IDF 벡터는 이미 길이가 1로 맞춰져 있어서 계산이 간단합니다.

| 단계 | 내용 |
|---|---|
| 1 | 예제 리뷰 유사도 표 |
| 2 | 한계 확인: 순서·반대말·동의어 |
| 3 | 비슷한 리뷰 찾기 |
| 4 | 검색어로 찾기 |

### 1단계. 예제 리뷰 유사도 표

In [87]:
# TF-IDF 도구와 유사도 계산 도구 불러오기
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 모든 절에서 함께 쓸 예제 리뷰 5개 (단원 2처럼 토큰화가 끝난 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

# 예제 리뷰를 TF-IDF 벡터로 바꾸기
tv = TfidfVectorizer(**opts)
X = tv.fit_transform(mini)
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬


# 리뷰 5개끼리 모두 유사도 계산해서 표로 출력
pd.DataFrame(cosine_similarity(X), index=mini, columns=[0, 1, 2, 3, 4]).round(2)

,0,1,2,3,4
배송 빠르다 포장 꼼꼼하다,1.00,0.62,0.62,0.00,0.00
배송 빠르다 포장 별로,0.62,1.00,1.00,0.00,0.00
포장 빠르다 배송 별로,0.62,1.00,1.00,0.00,0.00
가격 저렴하다 품질 좋다,0.00,0.00,0.00,1.00,0.57
가격 비싸다 품질 안 좋다,0.00,0.00,0.00,0.57,1.00


### 2단계. 한계 확인





**동의어 문제도 확인해 봅니다.** 뜻은 비슷한데 단어가 다르면?

In [104]:
# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 뜻이 비슷하지만 겹치는 단어가 없는 두 문장
pair = CountVectorizer(**opts).fit_transform(["향 은은하다", "냄새 은근하다"])

# 유사도 계산
print("유사도:", cosine_similarity(pair[0], pair[1])[0, 0])

유사도: 0.0


✅ 사람이 보기엔 비슷한 말인데 유사도는 **0**입니다. 지금까지 배운 방법은 모두 **단어의 뜻을 모릅니다.**
이 문제는 **단원 5 임베딩**에서 해결합니다.

### 3단계. 비슷한 리뷰 찾기
**✏️ [입력]** `query_index` 숫자를 바꿔 다른 리뷰로 찾아보세요.

In [105]:
# ===== ✏️ [입력] 기준이 될 리뷰 번호 =====
query_index = 2                                   # 비교 기준으로 쓸 리뷰의 위치 번호 (0부터 시작)
# 기준 리뷰 출력
print("기준 리뷰:", docs[query_index])             # docs에서 2번 위치(세 번째) 리뷰를 출력함
print()                                           # 빈 줄 하나를 출력함
# ========================================

# 띄어쓰기 기준으로만 단어를 나누고, 영어 대소문자를 그대로 둠
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 실제 데이터를 TF-IDF 벡터로 바꾸기 (5개 미만 리뷰에 나온 단어 제외)
tv_real = TfidfVectorizer(min_df=5, **opts)
# min_df=5 : 5개 미만의 리뷰에만 나온 단어는 사전에서 뺌
# **opts   : 공통 설정(공백 기준 토큰화, 대소문자 유지)을 펼쳐 넣음
X_real = tv_real.fit_transform(docs)  #docs =엑셀파일 또는 mini X 20
# 사전을 만들고, 각 리뷰를 TF-IDF 점수 벡터로 바꿈 → 행 = 리뷰, 열 = 단어
#fit(문서들)	단어 사전만 만듦	학습이 끝난 vectorizer 자신
#transform(문서들)	이미 만든 사전으로 문서를 숫자로 바꿈	문서-단어 행렬
#fit_transform(문서들)	사전을 만들고, 같은 문서를 바로 숫자로 바꿈	문서-단어 행렬

# 기준 리뷰와 모든 리뷰의 유사도 계산
scores = cosine_similarity(X_real[query_index], X_real)[0]
# X_real[query_index]           : 기준 리뷰 한 줄 (1 × 단어수)
# cosine_similarity(A, B)       : A의 각 행과 B의 각 행 사이 코사인 유사도 표를 만듦 → 1 × 리뷰수
# [0]                           : 그 표의 첫 번째(유일한) 행 → 길이 = 리뷰수인 1차원 배열

# 유사도 큰 순서로 정렬해서 자기 자신(첫 번째)을 빼고 5개 출력
for i in scores.argsort()[::-1][1:6]:
    # argsort()  : 유사도가 작은 순서대로의 리뷰 번호
    # [::-1]     : 뒤집어서 큰 순서대로
    # [1:6]      : 1번째부터 5번째까지 (0번째 = 가장 큰 것을 건너뜀)
    print(round(scores[i], 2), "|", docs[i])  #docs =엑셀파일 또는 mini X 20
    # round(값, 2) : 소수 둘째 자리까지 반올림해 출력함

기준 리뷰: 남편 선물 사다 향기 좋다 지인 추천 택배 상자 안 상자 찌그러지다 오다 깨지다 않다 하다 알리다 요즘 택배 조금 일 있다 정도 양호 감사 쓰다 구매

0.53 | 만점 배송 느리다 배송 상태 미흡 택배 상자 배송 제품 포장 없다 택배 상자 옆 면 찢어지다 위험 제품 겉면 일부 찌그러지다 속상하다 제품 만족 향 좋다 거품 질감 좋다 면도 후 약간 자극 있다 로션 바르다 되다
0.35 | 배송 오다 같다 상자 망가지다 오다 제품 사용 피부 맞다 않다 자극
0.35 | 크다 로션 정도 생각 주문 택배 상자 열다 엄청나다 용량 놀라다 보습 약하다 같다 로션 바르다 바르다 바르다 사용 피부 좋다 앞 이렇다 쓰다
0.29 | 남편 좋아하다 제품 선물 구입 향 좋다 발림 좋다 같다 선물 상자 요청 동봉 배송 되다 좋다 oo 상품 믿다 구매 있다 좋다 샘플 챙기다 기분 좋다
0.27 | 택배 포장 제품 상태 양호 비교 곳 비하다 가격 싸다 같다 사용 트러블 올라오다 피부 안 맞다


### 4단계. 검색어로 찾기
새 문장을 같은 사전으로 바꾸면 **검색어**처럼 쓸 수 있습니다.
**✏️ [입력]** `query`를 내 데이터에 맞는 단어로 바꿔 보세요. (단원 2처럼 `좋다`, `빠르다` 같은 기본형으로 입력)

In [107]:
# ===== ✏️ [입력] 검색어 =====
query = "배송 빠르다"                      # 찾고 싶은 내용을 토큰 형태(기본형, 공백 구분)로 입력함
# ============================

# 검색어를 같은 사전으로 벡터로 바꾸기
query_vec = tv_real.transform([query])
# [query]          : 문자열 하나를 리스트에 담음 (transform은 문서 "목록"을 받기 때문)
# transform(...)   : 앞 셀에서 fit한 사전과 IDF로 검색어를 TF-IDF 벡터로 바꿈 → 1 × 단어수

# 검색어와 모든 리뷰의 유사도 계산
scores = cosine_similarity(query_vec, X_real)[0]
# 검색어 벡터와 각 리뷰 벡터 사이 코사인 유사도 → 길이 = 리뷰수인 1차원 배열

# 유사도 큰 순서로 5개 출력
for i in scores.argsort()[::-1][:5]:
    # argsort()[::-1] : 유사도 큰 순서의 리뷰 번호
    # [:5]            : 앞에서 5개 (검색어는 리뷰 목록에 없으므로 건너뛸 것이 없음)
    print(round(scores[i], 2), "|", docs[i])  #docs =엑셀파일 또는 mini X 20

0.61 | 사용 보습 끈적이다 않다 좋다 배송 빠르다
0.59 | 머스크 향 좋다 흡수 빠르다 자극 좋다 배송 빠르다 할인 저렴하다 사다
0.56 | 가격 저렴하다 배송 빠르다 바르다 향 좋다 강력 추천
0.56 | 향 괜찮다 가격 저렴하다 쓰다 좋다 같다 배송 빠르다
0.55 | 배송 빠르다 가격 저렴하다 쓰다 번들거리다 않다 좋다


⚠️ **미리 알아두기:** 지금은 전체 데이터로 사전을 만들었습니다(`fit`). 실제 분류에서는 **학습용 데이터로만** 사전을 만들어야 합니다. 시험용 데이터 정보가 미리 섞이면(데이터 누수) 성능이 실제보다 좋게 나오기 때문입니다.


### ✅ 3-5 한 줄 정리
**리뷰를 화살표로 보고 방향이 비슷한지 재면 비슷한 리뷰를 찾을 수 있다.** 하지만 **단어의 뜻**은 모른다.

### ➡️ 다음 절로
3-6절에서 지금까지 배운 방법을 **한 표로 비교**하고 정리합니다.

---
## 3-6. 텍스트 표현 방법 종합 비교

### 🎯 이 절에서 배울 것
- 다섯 방법의 특징을 한눈에 비교
- 같은 리뷰 쌍이 방법마다 얼마나 다르게 계산되는지
- 상황에 맞는 방법 고르기

###  특징 비교표
| 방법 | 무엇을 표현 | 값 | 순서 | 흔한 단어 처리 | 단어의 뜻 |
|---|---|---|---|---|---|
| One-hot | 단어 | 0/1 (1이 하나) | ✕ | ✕ | ✕ |
| Binary BoW | 리뷰 | 0/1 (나왔나) | ✕ | ✕ | ✕ |
| BoW (개수) | 리뷰 | 나온 횟수 | ✕ | ✕ (흔할수록 큼) | ✕ |
| N-gram BoW | 리뷰 | 묶음 횟수 | △ (가까운 순서만) | ✕ | ✕ |
| TF-IDF | 리뷰 | 점수 | ✕ (N-gram과 쓰면 △) | ○ (흔할수록 작음) | ✕ |
| 임베딩 (단원 5) | 단어·문장 | 실수 | 방법에 따라 다름 | — | ○ |

### 어떤 방법을 고를까?
| 상황 | 먼저 써 볼 방법 | 이유 |
|---|---|---|
| 리뷰가 아주 짧음 | Binary BoW | 공식 문서: 짧은 글은 "나왔나/안 나왔나"가 더 안정적일 수 있음 |
| 키워드 뽑기·검색 | TF-IDF | 흔한 단어 점수를 낮춤 |
| 부정 표현이 중요 (감성분석) | TF-IDF + 1~2단어 묶음 | `안 좋다`를 살림 |
| 비슷한 말·문맥 이해가 중요 | 임베딩 (단원 5) | 단어의 뜻을 담음 |

> 이 표는 **출발점**입니다. 실제로 어느 방법이 좋은지는 데이터와 목적에 따라 다르므로,  분류 성능으로 직접 비교합니다.





**참고 공식 문서**
- [scikit-learn 텍스트 특징 추출](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
- [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) · [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) · [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)